# Stock Price Prediction Using Machine Learning

---

## AI/ML Engineering Internship — Task 2

| Field | Detail |
|---|---|
| **Organization** | DevelopersHub Corporation |
| **Role** | AI/ML Engineering Intern |
| **Task** | Task 2 — Stock Price Prediction |
| **Author** | Shahab Ahmed |
| **Date** | June 2026 |

---

## Problem Statement

Predicting stock prices is a fundamental challenge in financial analytics. Accurate forecasts enable investors to make informed decisions and optimize portfolio returns.

This project focuses on predicting the **next day's closing price** of **Apple Inc. (AAPL)** stock using classical machine learning regression models.

## Objectives

1. Acquire historical AAPL stock data (3+ years) using `yfinance`
2. Perform comprehensive Exploratory Data Analysis (EDA)
3. Engineer meaningful features from raw price and volume data
4. Train and evaluate **Linear Regression** and **Random Forest Regressor** models
5. Compare model performance using MAE, RMSE, and R² Score
6. Visualize predictions against actual prices

---
## Cell 2 — Install & Import Libraries

In [ ]:
import sys
!{sys.executable} -m pip install --quiet yfinance pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully.')

---
## Cell 3 — Download Stock Data

In [ ]:
ticker = 'AAPL'
start_date = '2021-01-01'
end_date = '2025-12-31'

df = yf.download(ticker, start=start_date, end=end_date, progress=False)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Downloaded {len(df)} trading days of {ticker} data.')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')

---
## Cell 4 — Dataset Overview

In [ ]:
print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')
print(f'\nMissing values per column:')
print(df.isnull().sum())
print(f'\nFirst 5 rows:')
display(df.head())

---
## Cell 5 — Statistical Summary

In [ ]:
print('=' * 60)
print('STATISTICAL SUMMARY')
print('=' * 60)
print('\nDataset Info:')
df.info()
print('\nDescriptive Statistics:')
display(df.describe())

---
## Cell 6 — Closing Price Trend

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df['Close'], color='#1f77b4', linewidth=1.2, label='Closing Price')
ax.fill_between(df.index, df['Close'], alpha=0.15, color='#1f77b4')
ax.set_title('AAPL Closing Price Trend', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Price (USD)', fontsize=13)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
## Cell 7 — Volume Analysis

In [ ]:
monthly_volume = df['Volume'].resample('ME').mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(monthly_volume.index, monthly_volume.values, width=20, color='#ff7f0e', alpha=0.8)
ax.set_title('AAPL Average Monthly Trading Volume', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Volume', fontsize=13)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x / 1e6:.0f}M'))
plt.tight_layout()
plt.show()

---
## Cell 8 — Moving Averages (20-day and 50-day)

In [ ]:
df['MA_20'] = df['Close'].rolling(window=20).mean()
df['MA_50'] = df['Close'].rolling(window=50).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df['Close'], label='Close', color='#1f77b4', linewidth=1.0, alpha=0.7)
ax.plot(df.index, df['MA_20'], label='20-Day MA', color='#ff7f0e', linewidth=1.5)
ax.plot(df.index, df['MA_50'], label='50-Day MA', color='#2ca02c', linewidth=1.5)
ax.set_title('AAPL Closing Price with Moving Averages', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Price (USD)', fontsize=13)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
## Cell 9 — Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[['Open', 'High', 'Low', 'Close', 'Volume']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, square=True)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## Cell 10 — Create Target Variable (Next Day's Close)

In [ ]:
df['Target'] = df['Close'].shift(-1)
df.dropna(subset=['Target'], inplace=True)

print("Target variable created: next day's Close price.")
print(f'Dataset shape after dropping last row: {df.shape}')
display(df[['Close', 'Target']].tail())

---
## Cell 11 — Feature Engineering

In [ ]:
df['DayOfWeek'] = df.index.dayofweek
df['DailyReturn'] = df['Close'].pct_change()
df['DailyReturn_Lag1'] = df['DailyReturn'].shift(1)
df['RollingMean_5'] = df['Close'].rolling(window=5).mean()
df['RollingMean_10'] = df['Close'].rolling(window=10).mean()
df['RollingStd_5'] = df['Close'].rolling(window=5).std()
df['High_Low_Range'] = df['High'] - df['Low']
df['Open_Close_Diff'] = df['Close'] - df['Open']
df['Volume_MA_5'] = df['Volume'].rolling(window=5).mean()

df.dropna(inplace=True)
df.drop(columns=['MA_20', 'MA_50'], inplace=True)

feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'DayOfWeek',
                'DailyReturn', 'DailyReturn_Lag1', 'RollingMean_5',
                'RollingMean_10', 'RollingStd_5', 'High_Low_Range',
                'Open_Close_Diff', 'Volume_MA_5']

print(f'Engineered {len(feature_cols)} features.')
print(f'Final dataset shape: {df.shape}')
display(df[feature_cols + ['Target']].tail())

---
## Cell 12 — Train/Test Split (Time-Based)

In [ ]:
split_index = int(len(df) * 0.8)

X = df[feature_cols]
y = df['Target']

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f'Training set: {X_train.shape[0]} samples ({df.index[0].date()} to {df.index[split_index - 1].date()})')
print(f'Test set:     {X_test.shape[0]} samples ({df.index[split_index].date()} to {df.index[-1].date()})')

---
## Cell 13 — Linear Regression Model

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_train_pred = lr_model.predict(X_train)
lr_test_pred = lr_model.predict(X_test)

print('Linear Regression — Training Complete')
print(f'Train R\u00b2: {r2_score(y_train, lr_train_pred):.4f}')
print(f'Test  R\u00b2: {r2_score(y_test, lr_test_pred):.4f}')

---
## Cell 14 — Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

print('Random Forest Regressor — Training Complete')
print(f'Train R\u00b2: {r2_score(y_train, rf_train_pred):.4f}')
print(f'Test  R\u00b2: {r2_score(y_test, rf_test_pred):.4f}')

---
## Cell 15 — Compare Both Models

In [ ]:
metrics = {
    'Metric': ['MAE', 'RMSE', 'R\u00b2 Score'],
    'Linear Regression': [
        mean_absolute_error(y_test, lr_test_pred),
        np.sqrt(mean_squared_error(y_test, lr_test_pred)),
        r2_score(y_test, lr_test_pred)
    ],
    'Random Forest': [
        mean_absolute_error(y_test, rf_test_pred),
        np.sqrt(mean_squared_error(y_test, rf_test_pred)),
        r2_score(y_test, rf_test_pred)
    ]
}

metrics_df = pd.DataFrame(metrics)
print('\nModel Comparison on Test Set:')
print('=' * 50)
display(metrics_df)

---
## Cell 16 — Actual vs Predicted Prices

In [ ]:
test_dates = df.index[split_index:]

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

axes[0].plot(test_dates, y_test.values, label='Actual', color='#1f77b4', linewidth=1.5)
axes[0].plot(test_dates, lr_test_pred, label='Linear Regression', color='#d62728', linewidth=1.2, alpha=0.8)
axes[0].set_title('Actual vs Predicted — Linear Regression', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(fontsize=11)

axes[1].plot(test_dates, y_test.values, label='Actual', color='#1f77b4', linewidth=1.5)
axes[1].plot(test_dates, rf_test_pred, label='Random Forest', color='#2ca02c', linewidth=1.2, alpha=0.8)
axes[1].set_title('Actual vs Predicted — Random Forest', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Price (USD)')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

---
## Cell 17 — MAE Calculation

In [ ]:
lr_mae = mean_absolute_error(y_test, lr_test_pred)
rf_mae = mean_absolute_error(y_test, rf_test_pred)

print(f'Linear Regression MAE: {lr_mae:.4f}')
print(f'Random Forest MAE:     {rf_mae:.4f}')
print(f'\nBest MAE: {"Random Forest" if rf_mae < lr_mae else "Linear Regression"}')

---
## Cell 18 — RMSE Calculation

In [ ]:
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_test_pred))
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))

print(f'Linear Regression RMSE: {lr_rmse:.4f}')
print(f'Random Forest RMSE:     {rf_rmse:.4f}')
print(f'\nBest RMSE: {"Random Forest" if rf_rmse < lr_rmse else "Linear Regression"}')

---
## Cell 19 — R² Score

In [ ]:
lr_r2 = r2_score(y_test, lr_test_pred)
rf_r2 = r2_score(y_test, rf_test_pred)

print(f'Linear Regression R\u00b2: {lr_r2:.4f}')
print(f'Random Forest R\u00b2:     {rf_r2:.4f}')
print(f'\nBest R\u00b2: {"Random Forest" if rf_r2 > lr_r2 else "Linear Regression"}')

---
## Cell 20 — Feature Importance

In [ ]:
importances = rf_model.feature_importances_
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='#2ca02c', edgecolor='white')
ax.set_title('Random Forest — Feature Importance', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Importance Score')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

---
## Cell 21 — Model Comparison Chart

In [ ]:
comparison_data = {
    'Metric': ['MAE', 'RMSE', 'R\u00b2 Score'],
    'Linear Regression': [lr_mae, lr_rmse, lr_r2],
    'Random Forest': [rf_mae, rf_rmse, rf_r2]
}
comp_df = pd.DataFrame(comparison_data).set_index('Metric')

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comp_df.index))
width = 0.35
bars1 = ax.bar(x - width / 2, comp_df['Linear Regression'], width, label='Linear Regression', color='#d62728', alpha=0.85)
bars2 = ax.bar(x + width / 2, comp_df['Random Forest'], width, label='Random Forest', color='#2ca02c', alpha=0.85)

ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(comp_df.index, fontsize=13)
ax.legend(fontsize=12)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

---
## Cell 22 — Key Insights

1. **Strong price autocorrelation** — Both models achieve high R² scores, indicating that historical price features are strong predictors of the next day's close.

2. **Random Forest outperforms Linear Regression** — The ensemble model captures non-linear relationships in the data, resulting in lower MAE and RMSE.

3. **Close price is the dominant feature** — Feature importance analysis confirms that the current day's closing price is the strongest predictor of tomorrow's close.

4. **Volume has limited predictive power** — Trading volume shows weak correlation with price movement, consistent with efficient market observations.

5. **Moving averages add value** — Engineered rolling mean features improve model performance by capturing short-term momentum.

---
## Cell 23 — Limitations

1. **No external features** — The model relies solely on historical price and volume data, ignoring macroeconomic indicators, earnings reports, and news sentiment.

2. **Single stock focus** — Results are specific to AAPL and may not generalize to other tickers without retraining.

3. **Time-based split only** — No cross-validation was performed due to the sequential nature of time-series data.

4. **No deep learning models** — LSTM, GRU, and Transformer architectures were not explored, which may capture temporal dependencies more effectively.

5. **Market regime changes** — The model may underperform during black-swan events or structural market shifts not present in training data.

---
## Cell 24 — Future Improvements

1. **Deep learning integration** — Implement LSTM and GRU networks for sequence-to-sequence price forecasting.

2. **Sentiment analysis** — Incorporate NLP-based sentiment scores from financial news, Twitter/X, and Reddit.

3. **Hyperparameter optimization** — Use Optuna or Bayesian optimization for automated hyperparameter tuning.

4. **Multi-stock portfolio** — Extend the pipeline to predict prices for a diversified portfolio of stocks.

5. **Real-time deployment** — Build a Streamlit or FastAPI dashboard for live predictions with streaming data.

6. **Walk-forward validation** — Implement expanding-window cross-validation for more robust evaluation.

---
## Cell 25 — Final Conclusion

This project demonstrates a complete end-to-end machine learning pipeline for stock price prediction. Both **Linear Regression** and **Random Forest Regressor** achieve strong predictive performance on AAPL data, with the Random Forest model delivering superior results across all evaluation metrics.

While the models show high R² scores, it is important to note that stock markets are inherently stochastic and influenced by numerous external factors. These models serve as a strong baseline and educational foundation, but should be combined with fundamental analysis and risk management strategies before being used for real trading decisions.

---

> *Task 2 completed as part of the AI/ML Engineering Internship at DevelopersHub Corporation.*